In [ ]:
# Mount Google Drive
from google.colab import drive, auth

# 1. Authenticate your Google account
# This will open a popup asking you to grant Colab access to your Google Cloud account
auth.authenticate_user()
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!apt-get -qq update
!apt-get -qq install mesa-opencl-icd libosmesa6 libosmesa6-dev
!pip -q install smplx pyopengl pyrender trimesh torchgeometry spacepy \
    git+https://github.com/mattloper/chumpy.git

# Clone SPIN and add to path
import sys, os
spin_path = '/content/SPIN'
if not os.path.exists(spin_path):
    !git clone -q https://github.com/avasenin-14/SPIN
if spin_path not in sys.path:
    sys.path.append(spin_path)

# Download SPIN pretrained data
%cd /content/SPIN
if not os.path.exists('data/model_checkpoint.pt'):
    !wget -q http://visiondata.cis.upenn.edu/spin/data.tar.gz && tar -xf data.tar.gz && rm data.tar.gz
    !wget -q http://visiondata.cis.upenn.edu/spin/model_checkpoint.pt -P data/
    !wget -q https://github.com/vchoutas/smplify-x/raw/master/smplifyx/prior.py -O smplify/prior.py

# Copy SMPL model files
import shutil
smpl_src = '/content/drive/MyDrive/dissertation/smpl/'
smpl_dst = './data/smpl/'
os.makedirs(smpl_dst, exist_ok=True)
for new_name, orig_name in {
    'SMPL_NEUTRAL.pkl': 'basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl',
    'SMPL_MALE.pkl':    'basicmodel_m_lbs_10_207_0_v1.1.0.pkl',
    'SMPL_FEMALE.pkl':  'basicmodel_f_lbs_10_207_0_v1.1.0.pkl',
}.items():
    src = smpl_src + orig_name
    dst = smpl_dst + new_name
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

print("Environment ready.")

In [ ]:
# Define your GCP Project ID and Bucket Name
from google.colab import auth

# 1. Authenticate your Google account
# This will open a popup asking you to grant Colab access to your Google Cloud account
auth.authenticate_user()

# 2. Define your GCP Project ID and Bucket Name
# Replace these with your actual project ID and bucket name
PROJECT_ID = "project-2f9fd1c8-00f8-4801-b48"
BUCKET_NAME = "artem-bucket-01"

# Set the active project for gcloud
!gcloud config set project {PROJECT_ID}

# 3. Download the dataset from GCS to local NVMe
print("Downloading images.tar.gz from GCS to local disk...")
print("This should take around 10 minutes at ~500MB/s+")

!gcloud --verbosity=info storage cp --no-clobber gs://{BUCKET_NAME}/images.tar.gz /content/images.tar.gz

print("Download complete! The dataset is ready at /content/images.tar.gz")

In [ ]:
!df -h

In [ ]:
import os
os.environ['PYOPENGL_PLATFORM'] = 'egl'

import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from torchvision.models import vit_b_16, ViT_B_16_Weights

from models.hmr import HMR, Bottleneck
from models.smpl import SMPL
from utils.geometry import rot6d_to_rotmat
from utils.renderer import Renderer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SMPL_MEAN_PARAMS = 'data/smpl_mean_params.npz'
CHECKPOINT_PATH  = 'data/model_checkpoint.pt'
FOCAL_LENGTH     = 5000
IMG_RES          = 224

In [ ]:
import tarfile, io
from torch.utils.data import IterableDataset, DataLoader

# The archives are now concatenated into a single file on the local disk.
# Point to this single, fast local file instead of the slow Drive splits.
ARCHIVES = ['/content/images.tar.gz']

# ViT preprocessing transform
vit_weights = ViT_B_16_Weights.DEFAULT
preprocess  = vit_weights.transforms()


class TarImageDataset(IterableDataset):
    """Stream images from one or more tar archives sequentially."""
    def __init__(self, archive_paths, transform=None):
        if isinstance(archive_paths, str):
            archive_paths = [archive_paths]
        self.archive_paths = archive_paths
        self.transform = transform

    def __iter__(self):
        for path in self.archive_paths:
            print(f"Streaming from {os.path.basename(path)} ...")
            with tarfile.open(path, mode='r|*') as tar:
                for member in tar:
                    if member.isfile() and member.name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        try:
                            f = tar.extractfile(member)
                            if f is not None:
                                img = Image.open(io.BytesIO(f.read())).convert('RGB')
                                if self.transform:
                                    img = self.transform(img)
                                yield img, member.name
                        except Exception as e:
                            print(f"Skipping {member.name}: {e}")


# Quick-check with first archive only
dataset    = TarImageDataset(ARCHIVES, transform=preprocess)
dataloader = DataLoader(dataset, batch_size=4, num_workers=0)

images, names = next(iter(dataloader))
images = images.to(device)
print(f"Batch shape: {images.shape}  |  Files: {names}")

In [ ]:
import json, zipfile, glob as _glob

# 1) H3.6M files already in SPIN data/ directory (from data.tar.gz download)
h36m_files = sorted(_glob.glob('data/*h36m*') + _glob.glob('data/**/*h36m*', recursive=True))
print("══ H3.6M files in SPIN data/ ══")
for f in h36m_files:
    sz = os.path.getsize(f) / 1e6
    print(f"  {f}  ({sz:.1f} MB)")
if not h36m_files:
    print("  (none)")

# 2) Inspect annotations.zip
ANN_ZIP = '/content/drive/MyDrive/dissertation/humans3.6m/annotations.zip'
print(f"\n══ {os.path.basename(ANN_ZIP)} ══")
with zipfile.ZipFile(ANN_ZIP, 'r') as z:
    names = z.namelist()
    exts = {}
    for n in names:
        ext = os.path.splitext(n)[1].lower() or '/'
        exts[ext] = exts.get(ext, 0) + 1
    print(f"  Entries: {len(names)},  Types: {exts}")
    for n in names[:20]:
        print(f"    {n}")

# 3) Bbox / root-joint JSON samples
BBOX_DIR = '/content/drive/MyDrive/dissertation/Bounding box + Root joint coordinate/Human3.6M'
if os.path.isdir(BBOX_DIR):
    for sub in sorted(os.listdir(BBOX_DIR)):
        jp = os.path.join(BBOX_DIR, sub, 'bbox_root_human36m_output.json')
        if os.path.exists(jp):
            with open(jp) as f:
                d = json.load(f)
            sample = d[0] if isinstance(d, list) else next(iter(d.values()))
            print(f"\n══ {sub} ══")
            print(f"  Type: {type(d).__name__}, Entries: {len(d)}, Sample: {sample}")

In [ ]:
# Initialize HMR (SPIN model)
spin_model = HMR(Bottleneck, [3, 4, 6, 3], SMPL_MEAN_PARAMS)

# Load pretrained weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
if 'model' in checkpoint:
    spin_model.load_state_dict(checkpoint['model'], strict=False)
else:
    print("⚠ Warning: 'model' key not found in checkpoint. Weights might not be loaded.")

# Move to GPU
spin_model.to(device)
spin_model.eval()

# Initialize SMPL layer (needed for loss calculation)
# We assume the model files are in 'data/smpl/'
smpl_layer = SMPL('data/smpl/', batch_size=1).to(device)

In [ ]:
# ViT as feature extractor (remove classification head)
vit_model = vit_b_16(weights=vit_weights)
vit_model.heads = nn.Identity()
vit_model.eval().to(device)

# Linear adapter: 768 → 2048 (untrained — random init)
adapter = nn.Sequential(
    nn.Linear(768, 1024),
    nn.GELU(),
    nn.Linear(1024, 2048)
).to(device)

In [ ]:
import json, zipfile, glob
from torchvision import transforms as T
import numpy as np
import os

def run_regressor(features_2048, spin_model):
    """Run the SPIN regressor head on 2048-d features (iterative regression)."""
    B = features_2048.shape[0]
    pred_pose  = spin_model.init_pose.expand(B, -1)
    pred_shape = spin_model.init_shape.expand(B, -1)
    pred_cam   = spin_model.init_cam.expand(B, -1)
    for _ in range(3):
        xc = torch.cat([features_2048, pred_pose, pred_shape, pred_cam], 1)
        xc = torch.relu(spin_model.fc1(xc))
        xc = spin_model.drop1(xc)
        xc = torch.relu(spin_model.fc2(xc))
        xc = spin_model.drop2(xc)
        pred_pose  = spin_model.decpose(xc)  + pred_pose
        pred_shape = spin_model.decshape(xc) + pred_shape
        pred_cam   = spin_model.deccam(xc)   + pred_cam
    return pred_pose, pred_shape, pred_cam


# ══════════════════════════════════════════════════════════════
# Load Ground-Truth 3D annotations
# ══════════════════════════════════════════════════════════════

ANN_ZIP   = '/content/drive/MyDrive/dissertation/humans3.6m/annotations.zip'
SPIN_H36M = 'data/dataset_extras/h36m_train.npz'


def load_annotations():
    """Load H3.6M annotations — tries SPIN .npz first, then JSON pairs from zip."""

    # ── Option A: SPIN-processed npz ──
    if os.path.exists(SPIN_H36M):
        print(f"Loading SPIN annotation file: {SPIN_H36M}")
        h36m = np.load(SPIN_H36M, allow_pickle=True)
        print(f"  Keys: {list(h36m.keys())}, Samples: {len(h36m['imgname'])}")
        gt_lookup = {str(name): i for i, name in enumerate(h36m['imgname'])}
        return dict(
            gt_lookup=gt_lookup,
            gt_j3d=h36m['S'],
            gt_center=h36m.get('center'),
            gt_scale=h36m.get('scale'),
        )

    # ── Option B: JSON annotations from zip ──
    extract_dir = '/content/h36m_annotations'
    if not os.path.exists(extract_dir) or not os.listdir(extract_dir):
        print(f"Extracting {ANN_ZIP} ...")
        with zipfile.ZipFile(ANN_ZIP, 'r') as z:
            z.extractall(extract_dir)

    # Look for *_data.json files to drive the loading
    # Structure: Subject -> data.json (metadata) + joint_3d.json (poses)
    data_files = sorted(glob.glob(f'{extract_dir}/annotations/*_data.json'))
    if not data_files:
        # Fallback for deep recursive search
        data_files = sorted(glob.glob(f'{extract_dir}/**/*_data.json', recursive=True))

    print(f"  Found {len(data_files)} subject data files.")

    gt_lookup = {}
    gt_j3d_list = []
    gt_bbox_list = []

    for df in data_files:
        base_name = os.path.basename(df)
        # Expect corresponding joint file: Human36M_subject1_data.json -> Human36M_subject1_joint_3d.json
        jf = df.replace('_data.json', '_joint_3d.json')

        if not os.path.exists(jf):
            print(f"  ⚠ No corresponding joints file for {base_name}, skipping.")
            continue

        print(f"  Processing {base_name} ...")

        with open(df, 'r') as f: d_data = json.load(f)
        with open(jf, 'r') as f: j_data = json.load(f)

        # Map image_id to bbox
        bbox_map = {ann['image_id']: ann['bbox'] for ann in d_data.get('annotations', [])}

        # Iterate images to link metadata to joints
        images = d_data.get('images', [])
        matched_count = 0

        for img in images:
            file_name = img.get('file_name')
            img_id = img.get('id')

            # Metadata keys (convert to str for JSON lookup)
            act = str(img.get('action_idx'))
            sub = str(img.get('subaction_idx'))
            cam = str(img.get('cam_idx'))

            # Frame index from filename (s_..._000001.jpg -> index 0)
            try:
                frame_str = os.path.splitext(file_name)[0].split('_')[-1]
                frame_idx = int(frame_str) - 1
            except (ValueError, IndexError):
                continue

            # Lookup joint: j_data[act][sub][cam_key][frame_idx]
            # Note: Camera keys in json might be 0-based ('0','1','2','3') while metadata is 1-based
            # We try exact match first, then 0-based conversion
            cam_key = cam
            if act in j_data and sub in j_data[act]:
                if cam not in j_data[act][sub]:
                    # Try converting 1 -> '0'
                    try:
                        cam_0 = str(int(cam) - 1)
                        if cam_0 in j_data[act][sub]:
                            cam_key = cam_0
                    except ValueError:
                        pass

            try:
                if act not in j_data or sub not in j_data[act] or cam_key not in j_data[act][sub]:
                    continue

                poses = j_data[act][sub][cam_key]
                # Convert to numpy to check structure
                poses_arr = np.array(poses, dtype=np.float32)

                # Handle flattened list (e.g. [Point1, Point2...] where Points are [x,y,z])
                # If shape is (TotalFrames*Joints, 3), we need to reshape.
                if poses_arr.ndim == 2 and poses_arr.shape[1] == 3:
                    # Heuristic: H3.6M usually has 32 joints (raw) or 17 joints.
                    if poses_arr.shape[0] % 32 == 0:
                         poses_arr = poses_arr.reshape(-1, 32, 3)
                    elif poses_arr.shape[0] % 17 == 0:
                         poses_arr = poses_arr.reshape(-1, 17, 3)
                    # If neither, maybe it's frame-wise already? (N, 3) where J=1? Unlikely.

                if 0 <= frame_idx < len(poses_arr):
                    kp3d = poses_arr[frame_idx]
                    # kp3d should be (J, 3)

                    idx = len(gt_j3d_list)
                    gt_lookup[str(file_name)] = idx
                    gt_j3d_list.append(kp3d)
                    gt_bbox_list.append(bbox_map.get(img_id))
                    matched_count += 1
            except (KeyError, IndexError, TypeError, ValueError):
                continue

        print(f"    -> Matched {matched_count} images")

    if not gt_j3d_list:
        print("  ⚠ Could not load any annotations.")
        return dict(gt_lookup={}, gt_j3d=None, gt_center=None, gt_scale=None)

    gt_j3d = np.stack(gt_j3d_list)
    print(f"  Total loaded: {len(gt_lookup)} annotations, joints shape: {gt_j3d.shape}")

    # Convert bbox [x,y,w,h] → center/scale
    has_bbox = any(b is not None for b in gt_bbox_list)
    gt_center, gt_scale = None, None
    if has_bbox:
        centers, scales = [], []
        for bbox in gt_bbox_list:
            if bbox is not None:
                x, y, w, h = bbox[:4]
                centers.append([x + w/2, y + h/2])
                scales.append(max(w, h) / 200.0)
            else:
                centers.append([0, 0])
                scales.append(1.0)
        gt_center = np.array(centers, dtype=np.float32)
        gt_scale  = np.array(scales, dtype=np.float32)
        print(f"  Bbox info available → crop enabled")

    return dict(gt_lookup=gt_lookup, gt_j3d=gt_j3d, gt_center=gt_center, gt_scale=gt_scale)


ann = load_annotations()
gt_lookup = ann['gt_lookup']
GT_J3D    = ann['gt_j3d']
GT_CTR    = ann['gt_center']
GT_SCL    = ann['gt_scale']
HAS_CROP  = GT_CTR is not None and GT_SCL is not None

print(f"\nGT annotations: {len(gt_lookup)} images, crop info: {HAS_CROP}")
if GT_J3D is not None:
    print(f"Joints shape: {GT_J3D.shape}")
    print(f"Sample image names: {list(gt_lookup.keys())[:3]}")


# ══════════════════════════════════════════════════════════════
# Supervised Dataset — stream tar, match with GT 3D joints
# ══════════════════════════════════════════════════════════════

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def crop_person(img_pil, center, scale):
    """Crop person ROI using center + scale."""
    s = float(scale) * 200
    cx, cy = float(center[0]), float(center[1])
    half = s / 2
    x1, y1 = int(max(0, cx - half)), int(max(0, cy - half))
    x2, y2 = int(min(img_pil.width, cx + half)), int(min(img_pil.height, cy + half))
    return img_pil.crop((x1, y1, x2, y2))


class H36MSupervisedDataset(IterableDataset):
    """Stream images from tar archives, pair with GT 3D joints."""
    def __init__(self, archive_paths, gt_lookup, gt_joints,
                 gt_center=None, gt_scale=None, transform=None):
        self.archive_paths = archive_paths if isinstance(archive_paths, list) else [archive_paths]
        self.gt_lookup = gt_lookup
        self.gt_joints = gt_joints
        self.gt_center = gt_center
        self.gt_scale  = gt_scale
        self.transform = transform
        self.do_crop   = gt_center is not None and gt_scale is not None

    def _match_name(self, tar_name):
        """Try to match tar entry name to GT lookup (handles path prefixes)."""
        if tar_name in self.gt_lookup:
            return self.gt_lookup[tar_name]
        # Try just the filename
        basename = os.path.basename(tar_name)
        if basename in self.gt_lookup:
            return self.gt_lookup[basename]
        # Try relative path without leading dirs
        parts = tar_name.replace('\\', '/').split('/')
        for i in range(len(parts)):
            candidate = '/'.join(parts[i:])
            if candidate in self.gt_lookup:
                return self.gt_lookup[candidate]
        return None

    def __iter__(self):
        matched, skipped = 0, 0
        for path in self.archive_paths:
            print(f"  Streaming {os.path.basename(path)} ...")
            try:
                with tarfile.open(path, mode='r|*') as tar:
                    for member in tar:
                        if not member.isfile():
                            continue
                        if not member.name.lower().endswith(('.png', '.jpg', '.jpeg')):
                            continue
                        idx = self._match_name(member.name)
                        if idx is None:
                            skipped += 1
                            continue
                        try:
                            f = tar.extractfile(member)
                            img = Image.open(io.BytesIO(f.read())).convert('RGB')
                            if self.do_crop:
                                img = crop_person(img, self.gt_center[idx], self.gt_scale[idx])
                            if self.transform:
                                img = self.transform(img)
                            j3d = self.gt_joints[idx]
                            if j3d.shape[-1] == 4:
                                j3d = j3d[:, :3]
                            j3d = torch.tensor(j3d, dtype=torch.float32)
                            yield img, j3d
                            matched += 1
                        except Exception:
                            continue
            except (EOFError, tarfile.ReadError) as e:
                print(f"  ⚠ Warning: corrupted archive {os.path.basename(path)} ({e}). Moving to next.")
            except Exception as e:
                print(f"  ⚠ Error opening {os.path.basename(path)}: {e}")

        print(f"  Epoch done — matched: {matched}, skipped (no GT): {skipped}")


# ══════════════════════════════════════════════════════════════
# Training setup
# ══════════════════════════════════════════════════════════════

# Freeze SPIN regressor head
# Unfreeze SPIN regressor head and collect parameters
regressor_params = []
for name, param in spin_model.named_parameters():
    if any(k in name for k in ['fc1', 'fc2', 'decpose', 'decshape', 'deccam',
                                 'init_pose', 'init_shape', 'init_cam']):
        param.requires_grad = False
        regressor_params.append(param)

# Freeze the entire ViT backbone to only train the adapter
for param in vit_model.parameters():
    param.requires_grad = False
vit_model.eval()

adapter = nn.Linear(768, 2048).to(device)
adapter = nn.Sequential(
    nn.Linear(768, 1024),
    nn.GELU(),
    nn.Linear(1024, 2048)
).to(device)

optimizer = torch.optim.AdamW(
    adapter.parameters(),
    lr=1e-4, weight_decay=1e-4,
)
optimizer = torch.optim.AdamW([
    {'params': adapter.parameters(), 'lr': 1e-4},
    {'params': regressor_params, 'lr': 1e-5}
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Dataloader (streams all archives, yields only images with GT)
train_loader = DataLoader(
    H36MSupervisedDataset(
        ARCHIVES, gt_lookup, GT_J3D,
        gt_center=GT_CTR if HAS_CROP else None,
        gt_scale=GT_SCL if HAS_CROP else None,
        transform=train_transform,
    ),
    batch_size=16, num_workers=0,
)

NUM_EPOCHS = 10
LOG_EVERY  = 50
SAVE_PATH  = '/content/drive/MyDrive/dissertation/vit_adapter_supervised_ckpt.pt'

print(f"\nTraining ViT + adapter for {NUM_EPOCHS} epochs (supervised, MPJPE loss)")
print(f"GT images: {len(gt_lookup)}, Crop: {HAS_CROP}")
print(f"Trainable params: ")
print(f"{sum(p.numel() for p in vit_model.parameters() if p.requires_grad) + sum(p.numel() for p in adapter.parameters()):,}")
print(f"{sum(p.numel() for p in spin_model.parameters() if p.requires_grad) + sum(p.numel() for p in adapter.parameters()):,}")

In [ ]:
import json
import os

base_dir = '/content/h36m_annotations/annotations'
files_to_check = [
    'Human36M_subject1_joint_3d.json',
    'Human36M_subject1_data.json'
]

for fname in files_to_check:
    fpath = os.path.join(base_dir, fname)
    if not os.path.exists(fpath):
        continue

    print(f"\n--- Inspecting {fname} ---")
    with open(fpath, 'r') as f:
        data = json.load(f)

    print(f"Type: {type(data)}")
    if isinstance(data, dict):
        print(f"Keys: {list(data.keys())}")
        # Print a sample of values for the first key
        first_key = list(data.keys())[0]
        print(f"Sample value for key '{first_key}': {str(data[first_key])[:200]}...")
    elif isinstance(data, list):
        print(f"Length: {len(data)}")
        print(f"First item: {data[0]}")

In [ ]:
import torch
import numpy as np

# Load H3.6M Joint Regressor (SMPL vertices -> 17 H3.6M joints)
J_REGRESSOR_PATH = 'data/J_regressor_h36m.npy'
if os.path.exists(J_REGRESSOR_PATH):
    j_regressor = np.load(J_REGRESSOR_PATH)
    j_regressor = torch.tensor(j_regressor, dtype=torch.float32).to(device)
    print(f"Loaded J_regressor_h36m: {j_regressor.shape}")
else:
    print("⚠ Warning: J_regressor_h36m.npy not found. Using default SMPL joints (might cause mismatch).")
    j_regressor = None

# H3.6M Raw (32) to Standard (17) indices
# Common mapping: Hip, RHip, RKnee, RAnkle, LHip, LKnee, LAnkle, Spine, Neck, Head, ...
H36M_TO_17 = [0, 1, 2, 3, 6, 7, 8, 12, 13, 14, 15, 17, 18, 19, 25, 26, 27]

history = {'epoch': [], 'loss': []}

# Quick sanity check settings
#STEPS_PER_EPOCH = 50
WARMUP_EPOCHS = 2
#print(f"Starting training loop (Sanity Check: {STEPS_PER_EPOCH} steps/epoch)...")

for epoch in range(1, NUM_EPOCHS + 1):
    # Unfreeze the regressor after warmup epochs
    if epoch == WARMUP_EPOCHS + 1:
        print("\n=> Warmup complete! Unfreezing SPIN regressor head...")
        for param in regressor_params:
            param.requires_grad = True

    epoch_loss, n_batches = 0.0, 0

    for step, (imgs, gt_j3d) in enumerate(train_loader, 1):
        # Stop after N steps for quick sanity check
        #if step > STEPS_PER_EPOCH:
        #    break

        imgs   = imgs.to(device)       # (B, 3, 224, 224)
        gt_j3d = gt_j3d.to(device)     # (B, J_gt, 3)

        # Get teacher features from the frozen ResNet backbone
        with torch.no_grad():
            x = spin_model.conv1(imgs)
            x = spin_model.bn1(x)
            x = spin_model.relu(x)
            x = spin_model.maxpool(x)
            x = spin_model.layer1(x)
            x = spin_model.layer2(x)
            x = spin_model.layer3(x)
            x = spin_model.layer4(x)
            x = spin_model.avgpool(x)
            target_feats = x.view(x.size(0), -1)
            _, _, target_cam = run_regressor(target_feats, spin_model)

        # Forward: ViT → adapter → SPIN regressor → SMPL
        feats    = vit_model(imgs)                                # (B, 768)
        feats_2k = adapter(feats)                                 # (B, 2048)
        pred_pose, pred_shape, pred_cam = run_regressor(feats_2k, spin_model)
        pred_rotmat = rot6d_to_rotmat(pred_pose).view(-1, 24, 3, 3)

        # Get SMPL output (vertices)
        smpl_out = smpl_layer(
            global_orient=pred_rotmat[:, 0].unsqueeze(1),
            body_pose=pred_rotmat[:, 1:],
            betas=pred_shape,
            pose2rot=False,
        )

        # ── Regress to H3.6M joints ──
        if j_regressor is not None:
            # vertices: (B, 6890, 3) x (17, 6890)^T -> (B, 17, 3)
            pred_j3d = torch.matmul(j_regressor, smpl_out.vertices)

            # Subset GT if it looks like Raw 32-joint skeleton
            if gt_j3d.shape[1] == 32:
                gt_target = gt_j3d[:, H36M_TO_17]
            elif gt_j3d.shape[1] == 17:
                gt_target = gt_j3d
            else:
                # Fallback: keep as is (e.g. 16 joints)
                gt_target = gt_j3d

            # Match prediction to GT size if they differ (e.g. 17 vs 16)
            if pred_j3d.shape[1] > gt_target.shape[1]:
                 pred_j3d = pred_j3d[:, :gt_target.shape[1]]
            elif pred_j3d.shape[1] < gt_target.shape[1]:
                 gt_target = gt_target[:, :pred_j3d.shape[1]]
        else:
            # Fallback to SMPL joints (topology mismatch likely!)
            pred_j3d = smpl_out.joints
            gt_target = gt_j3d[:, :pred_j3d.shape[1]]

        # ── Fix Units: SMPL (m) -> H3.6M (mm) ──
        pred_j3d = pred_j3d * 1000.0

        # Root-relative alignment (subtract pelvis)
        # For H3.6M, index 0 is typically Pelvis/Hip
        pred_j3d_rel = pred_j3d - pred_j3d[:, [0]]
        gt_j3d_rel   = gt_target - gt_target[:, [0]]

        # MPJPE loss
        mpjpe_loss = torch.sqrt(((pred_j3d_rel - gt_j3d_rel) ** 2).sum(-1) + 1e-8).mean()
        distill_loss = torch.nn.functional.mse_loss(feats_2k, target_feats)
        cam_loss = torch.nn.functional.mse_loss(pred_cam, target_cam)
        loss = mpjpe_loss + 0.1 * distill_loss + 0.1 * cam_loss # Blend MPJPE, Feature, and Camera Distillation

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches  += 1

        if step % 10 == 0:  # Log more frequently for short epochs
            print(f"  epoch {epoch} | step {step:>3d} | Loss {loss.item():.1f}")

    scheduler.step()
    avg = epoch_loss / max(n_batches, 1)
    history['epoch'].append(epoch)
    history['loss'].append(avg)
    print(f"Epoch {epoch}/{NUM_EPOCHS}  avg_Loss={avg:.1f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}")

# Save checkpoint
torch.save({
    'vit_state_dict':     vit_model.state_dict(),
    'adapter_state_dict': adapter.state_dict(),
    'optimizer':          optimizer.state_dict(),
    'history':            history,
}, SAVE_PATH)
print(f"\nCheckpoint saved → {SAVE_PATH}")

In [ ]:
# --- Training curve ---
plt.figure(figsize=(8, 4))
# Note: loss is already in mm from the training loop, so we don't multiply by 1000 here
plt.plot(history['epoch'], history['loss'], marker='o')
plt.xlabel('Epoch'); plt.ylabel('MPJPE (mm)'); plt.title('Supervised Training — MPJPE')
plt.grid(True); plt.tight_layout(); plt.show()

# --- Re-evaluate on the quick-check batch from earlier ---
vit_model.eval()
with torch.no_grad():
    # 'images' is the batch (B, 3, 224, 224) defined in cell e510062d
    feats     = vit_model(images)
    feats_2k  = adapter(feats)
    ft_pose, ft_shape, ft_cam = run_regressor(feats_2k, spin_model)
    ft_rotmat = rot6d_to_rotmat(ft_pose).view(-1, 24, 3, 3)

# Get SMPL output
out_ft = smpl_layer(
    global_orient=ft_rotmat[:, 0].unsqueeze(1),
    body_pose=ft_rotmat[:, 1:],
    betas=ft_shape,
    pose2rot=False,
)

# --- Visualization Helper ---
def cam_to_translation(cam, focal_length=5000., img_res=224):
    """Convert weak perspective camera (s, tx, ty) to 3D translation (tx, ty, tz)."""
    s, tx, ty = cam
    tz = 2 * focal_length / (img_res * s + 1e-9)
    return np.array([tx, ty, tz])

# Instantiate renderer if needed
if 'renderer' not in locals():
    from utils.renderer import Renderer
    # Handle faces being tensor or numpy
    faces = smpl_layer.faces
    if isinstance(faces, torch.Tensor):
        faces = faces.cpu().numpy()

    renderer = Renderer(focal_length=FOCAL_LENGTH, img_res=IMG_RES, faces=faces)

# Prepare first image for display
img_t = images[0].cpu()
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_np = (img_t * std + mean).numpy().transpose(1, 2, 0)
img_np = np.clip(img_np, 0, 1)
img_bgr = (img_np * 255).astype(np.uint8)[:, :, ::-1]

# Render Prediction
verts_ft = out_ft.vertices[0].detach().cpu().numpy()
trans_ft = cam_to_translation(ft_cam[0].detach().cpu().numpy(), FOCAL_LENGTH, IMG_RES)
render_ft = renderer(verts_ft, trans_ft, img_bgr.copy())

# Show Side-by-Side
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.title("Input Image")
plt.axis('off')

plt.subplot(1, 2, 2)
# Ensure uint8 for display
plt.imshow(render_ft.astype(np.uint8)[:, :, ::-1])
plt.title("Prediction (ViT + Adapter)")
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Prepare Input Image
# We use the first image from the batch we loaded earlier
idx = 0
img_tensor = images[idx].unsqueeze(0).to(device)  # (1, 3, 224, 224)

# Helper to convert tensor back to displayable image
def tensor_to_img(t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(t.device)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(t.device)
    img = (t * std + mean).permute(1, 2, 0).cpu().numpy()
    return np.clip(img, 0, 1)

img_display = tensor_to_img(img_tensor[0])
img_bgr = (img_display * 255).astype(np.uint8)[:, :, ::-1]  # RGB -> BGR for Renderer

# 2. Inference: Standard SPIN (ResNet backbone)
# SPIN has a .forward() method that uses its internal ResNet
with torch.no_grad():
    pred_rotmat_spin, pred_betas_spin, pred_cam_spin = spin_model(img_tensor)

# 3. Inference: ViT + Adapter + SPIN Head
# We manually feed ViT feats -> Adapter -> SPIN Regressor
with torch.no_grad():
    vit_feats = vit_model(img_tensor)
    adapter_feats = adapter(vit_feats)
    pred_pose_vit, pred_betas_vit, pred_cam_vit = run_regressor(adapter_feats, spin_model)
    pred_rotmat_vit = rot6d_to_rotmat(pred_pose_vit).view(-1, 24, 3, 3)

# 4. Get SMPL Meshes
# Standard SPIN output
out_spin = smpl_layer(
    global_orient=pred_rotmat_spin[:, 0].unsqueeze(1),
    body_pose=pred_rotmat_spin[:, 1:],
    betas=pred_betas_spin,
    pose2rot=False,
)
# ViT output
out_vit = smpl_layer(
    global_orient=pred_rotmat_vit[:, 0].unsqueeze(1),
    body_pose=pred_rotmat_vit[:, 1:],
    betas=pred_betas_vit,
    pose2rot=False,
)

# 5. Render
# Ensure renderer exists
if 'renderer' not in locals():
    from utils.renderer import Renderer
    # Handle faces being tensor or numpy
    faces = smpl_layer.faces
    if isinstance(faces, torch.Tensor):
        faces = faces.cpu().numpy()
    renderer = Renderer(focal_length=FOCAL_LENGTH, img_res=IMG_RES, faces=faces)

def cam_to_trans(cam, force_visible=False, z_dist=5.0):
    s, tx, ty = cam

    if force_visible:
        # Force centering
        return np.array([0.0, 0.0, z_dist])
    else:
        tz = 2 * FOCAL_LENGTH / (IMG_RES * s + 1e-9)
        return np.array([tx, ty, tz])

    return np.array([tx, ty, tz])

# Render SPIN (Trust prediction)
verts_spin = out_spin.vertices[0].detach().cpu().numpy()
trans_spin = cam_to_trans(pred_cam_spin[0].detach().cpu().numpy(), force_visible=False)
rend_spin  = renderer(verts_spin, trans_spin, img_bgr.copy())

# Render ViT (Force visibility)
verts_vit = out_vit.vertices[0].detach().cpu().numpy()
# First try: Z=5m
trans_vit = cam_to_trans(pred_cam_vit[0].detach().cpu().numpy(), force_visible=True, z_dist=5.0)
rend_vit  = renderer(verts_vit, trans_vit, img_bgr.copy())

# Render ViT (Predicted Camera)
trans_vit_pred = cam_to_trans(pred_cam_vit[0].detach().cpu().numpy(), force_visible=False)
rend_vit_pred = renderer(verts_vit, trans_vit_pred, img_bgr.copy())

# 6. Check for missing projection
diff = np.abs(rend_vit.astype(int) - img_bgr.astype(int)).sum()
print(f"Difference between Input and ViT Render (z=5m): {diff}")

if diff < 1000: # Arbitrary small threshold
    print("⚠ Projection missing at 5m! Trying z=50m ...")
    trans_vit = cam_to_trans(pred_cam_vit[0].detach().cpu().numpy(), force_visible=True, z_dist=50.0)
    rend_vit  = renderer(verts_vit, trans_vit, img_bgr.copy())

    diff2 = np.abs(rend_vit.astype(int) - img_bgr.astype(int)).sum()
    print(f"Difference at z=50m: {diff2}")

# 7. Plot
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(img_display)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 4, 2)
# Ensure uint8
plt.imshow(rend_spin.astype(np.uint8)[:, :, ::-1])
plt.title("Standard SPIN Prediction")
plt.axis('off')

plt.subplot(1, 4, 3)
# Ensure uint8
plt.imshow(rend_vit.astype(np.uint8)[:, :, ::-1])
plt.title("Ours (ViT) [Forced Center]")
plt.axis('off')

plt.subplot(1, 4, 4)
# Ensure uint8
plt.imshow(rend_vit_pred.astype(np.uint8)[:, :, ::-1])
plt.title("Ours (ViT) [Predicted Camera]")
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Compare raw numbers from SPIN and ViT+Adapter
print("--- Camera Parameters (Scale, Tx, Ty) ---")
print(f"SPIN Camera: {pred_cam_spin[0].detach().cpu().numpy()}")
print(f"ViT Camera:  {pred_cam_vit[0].detach().cpu().numpy()}")
print()

print("--- Shape Parameters (Betas) ---")
print(f"SPIN Betas (first 5): {pred_betas_spin[0][:5].detach().cpu().numpy()}")
print(f"ViT Betas (first 5):  {pred_betas_vit[0][:5].detach().cpu().numpy()}")
print()

# Calculate Mean Per-Vertex Error between the two predictions
diff_vertices = np.sqrt(((verts_spin - verts_vit) ** 2).sum(axis=-1)).mean()
print(f"Mean Per-Vertex Difference between SPIN and ViT: {diff_vertices:.4f} m")